In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/yakshithm/project-1/sample_submission (1).csv
/kaggle/input/datasets/yakshithm/project-1/train.csv
/kaggle/input/datasets/yakshithm/project-1/test.csv


In [2]:
import csv
import pandas as pd
import networkx as nx

def parse_adjacency_list(file_path):
    print("Parsing variable-length adjacency list...")
    edges = []

    # Use standard Python I/O to handle variable row lengths safely
    with open(file_path, 'r') as file:
        # If your data is space or tab separated instead of comma, change delimiter (e.g., delimiter='\t')
        reader = csv.reader(file, delimiter=',')

        for row in reader:
            if not row:
                continue

            source = row[0]
            targets = row[1:] # Everything after the first element is a connection

            for target in targets:
                target = target.strip()
                if target: # Skip empty strings caused by trailing commas
                    edges.append({'source': source, 'target': target})

    print(f"Successfully extracted {len(edges)} individual edges.")
    return pd.DataFrame(edges)

# 1. Flatten the broken CSV into a clean DataFrame
df_clean = parse_adjacency_list('/kaggle/input/datasets/yakshithm/project-1/train.csv')

# 2. Now you can safely load it into NetworkX
G_directed = nx.from_pandas_edgelist(df_clean, 'source', 'target', create_using=nx.DiGraph())
print(f"Graph loaded: {G_directed.number_of_nodes()} nodes, {G_directed.number_of_edges()} edges.")

Parsing variable-length adjacency list...
Successfully extracted 24003361 individual edges.
Graph loaded: 4867136 nodes, 23945602 edges.


In [ ]:
#1

import pandas as pd
import networkx as nx
import numpy as np
import scipy.sparse as sp
import random
from tqdm import tqdm # For progress bars

def build_feature_dataset(G_dir, num_samples=100000):
    """
    Generates a balanced dataset of real and fake edges, then extracts 
    topological and global features using optimized sparse matrix math.
    """
    print(f"--- Starting Feature Extraction for {num_samples} Positive & Negative Samples ---")
    
    nodes = list(G_dir.nodes())
    edges = list(G_dir.edges())
    
    # ---------------------------------------------------------
    # 1. GENERATE POSITIVE & NEGATIVE SAMPLES
    # ---------------------------------------------------------
    print("Sampling positive edges...")
    # Randomly select a subset of existing edges
    positive_edges = random.sample(edges, min(num_samples, len(edges)))
    
    print("Generating negative edges (fake links)...")
    negative_edges = set()
    while len(negative_edges) < len(positive_edges):
        u = random.choice(nodes)
        v = random.choice(nodes)
        if u != v and not G_dir.has_edge(u, v):
            negative_edges.add((u, v))
    negative_edges = list(negative_edges)
    
    # Combine and create labels (1 = Real, 0 = Fake)
    all_edges = positive_edges + negative_edges
    labels = [1] * len(positive_edges) + [0] * len(negative_edges)
    
    # ---------------------------------------------------------
    # 2. PRECOMPUTE GLOBAL GRAPH METRICS
    # ---------------------------------------------------------
    print("Precomputing PageRank and Degrees (This takes a moment)...")
    pagerank_dict = nx.pagerank(G_dir, alpha=0.85)
    in_degrees = dict(G_dir.in_degree())
    out_degrees = dict(G_dir.out_degree())
    
    # ---------------------------------------------------------
    # 3. PREPARE SPARSE MATRIX FOR FAST TOPOLOGICAL MATH
    # ---------------------------------------------------------
    print("Building Scipy Sparse Matrices for topology...")
    # Map node IDs to matrix indices
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    # Create undirected sparse matrix for mutual friends logic
    A_dir = nx.to_scipy_sparse_array(G_dir, nodelist=nodes, format='csr')
    A_undir = A_dir + A_dir.T
    A_undir.data = np.ones_like(A_undir.data) # Binarize to 1s
    
    # Precompute Adamic-Adar penalty weights: w = 1 / log(degree)
    # Adding 2 to denominator to avoid division by zero or negative logs
    degrees_array = np.array(A_undir.sum(axis=1)).flatten()
    aa_weights = 1.0 / np.log(degrees_array + 2) 
    
    # ---------------------------------------------------------
    # 4. EXTRACT FEATURES
    # ---------------------------------------------------------
    print("Extracting pairwise features using matrix intersections...")
    features = []
    
    for i in tqdm(range(len(all_edges))):
        u, v = all_edges[i]
        label = labels[i]
        
        idx_u = node_to_idx[u]
        idx_v = node_to_idx[v]
        
        # Get sparse row vectors for u and v
        row_u = A_undir[idx_u]
        row_v = A_undir[idx_v]
        
        # MATH: Dot product of two binarized rows equals their intersection (Common Neighbors)
        common_neighbors = row_u.multiply(row_v)
        cn_count = common_neighbors.sum()
        
        # Jaccard = Intersection / (Degree U + Degree V - Intersection)
        deg_u = degrees_array[idx_u]
        deg_v = degrees_array[idx_v]
        union_size = deg_u + deg_v - cn_count
        jaccard = cn_count / union_size if union_size > 0 else 0
        
        # Adamic-Adar = Sum of (1 / log(degree)) for all common neighbors
        # We find the non-zero indices of the intersection and sum their precomputed weights
        shared_node_indices = common_neighbors.indices
        adamic_adar = np.sum(aa_weights[shared_node_indices])
        
        # Preferential Attachment = Degree U * Degree V
        pref_attach = deg_u * deg_v
        
        # Compile row
        features.append({
            'source': u,
            'target': v,
            'label': label,
            'common_neighbors': cn_count,
            'jaccard': jaccard,
            'adamic_adar': adamic_adar,
            'pref_attachment': pref_attach,
            'pr_source': pagerank_dict.get(u, 0),
            'pr_target': pagerank_dict.get(v, 0),
            'in_deg_target': in_degrees.get(v, 0),
            'out_deg_source': out_degrees.get(u, 0)
        })
        
    print("Compilation complete.")
    return pd.DataFrame(features)

# --- EXECUTION ---
# Assuming G_directed is already in memory from your previous parsing step
# Extracting features for 100k positive + 100k negative edges = 200k total rows
df_training_data = build_feature_dataset(G_directed, num_samples=100000)

# Save to CSV for ML training
df_training_data.to_csv('link_prediction_training_data.csv', index=False)
print(df_training_data.head())

In [ ]:
df_training_data.shape

In [ ]:
df_training_data[df_training_data['label'] == 0].head()

In [ ]:
!pip uninstall -y community
!pip uninstall -y python-louvain
!pip install python-louvain

In [4]:
#2

import random
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm

# ───────────────────────────────────────────────
# Balanced Directed Sampling
# ───────────────────────────────────────────────
def generate_directed_samples(G, num_samples=100_000, seed=42):
    random.seed(seed)
    nodes = list(G.nodes())
    all_edges = list(G.edges())

    pos_count = min(num_samples, len(all_edges))
    positive_samples = random.sample(all_edges, pos_count)

    neg_samples = set()
    while len(neg_samples) < pos_count:
        u, v = random.sample(nodes, 2)
        if not G.has_edge(u, v) and not G.has_edge(v, u):
            neg_samples.add((u, v))

    return positive_samples, list(neg_samples)

# ───────────────────────────────────────────────
# Memory-Safe Feature Extraction
# ───────────────────────────────────────────────
def extract_features(sample, G):
    u, v, label = sample

    features = {
        "source": u,
        "target": v,
        "label": label,
    }

    # Directed neighborhoods
    u_in = set(G.predecessors(u))
    u_out = set(G.successors(u))
    v_in = set(G.predecessors(v))
    v_out = set(G.successors(v))

    u_all = u_in | u_out
    v_all = v_in | v_out

    # Degree features
    features["u_d_in"] = len(u_in)
    features["u_d_out"] = len(u_out)
    features["v_d_in"] = len(v_in)
    features["v_d_out"] = len(v_out)

    features["u_d"] = len(u_all)
    features["v_d"] = len(v_all)

    # Overlap features
    common = u_all & v_all
    union = u_all | v_all

    features["common_friends"] = len(common)
    features["jaccard_coeff"] = len(common) / len(union) if len(union) > 0 else 0

    features["transitive_uv"] = len(u_out & v_in)
    features["transitive_vu"] = len(v_out & u_in)

    features["pref_attachment"] = features["u_d"] * features["v_d"]
    features["opposite_direction"] = 1 if G.has_edge(v, u) else 0

    return features

# ───────────────────────────────────────────────
# Main Builder (Safe Version)
# ───────────────────────────────────────────────
def build_directed_dataset_safe(
    G,
    num_samples=50_000,
    compute_pagerank=True,
    seed=42
):

    random.seed(seed)
    np.random.seed(seed)

    print("=== Memory-Safe Directed Link Prediction ===")

    # Sampling
    pos_samples, neg_samples = generate_directed_samples(G, num_samples, seed)
    all_samples = [(u, v, 1) for u, v in pos_samples] + \
                  [(u, v, 0) for u, v in neg_samples]

    random.shuffle(all_samples)

    # Optional PageRank
    pagerank = None
    if compute_pagerank:
        print("Computing PageRank...")
        pagerank = nx.pagerank(G, alpha=0.85)

    # Feature extraction (NO multiprocessing)
    print("Extracting features...")
    rows = []
    for sample in tqdm(all_samples):
        row = extract_features(sample, G)

        if pagerank:
            row["pr_source"] = pagerank.get(row["source"], 0)
            row["pr_target"] = pagerank.get(row["target"], 0)

        rows.append(row)

    df = pd.DataFrame(rows)

    print(f"Dataset ready → {df.shape[0]:,} rows × {df.shape[1]:,} columns")

    return df

# ───────────────────────────────────────────────
# Example Usage
# ───────────────────────────────────────────────
if __name__ == "__main__":

    df = build_directed_dataset_safe(
        G_directed,
        num_samples=100_000,   # start small
        compute_pagerank=True,
        seed=42
    )

    df.to_csv("link_prediction_safe.csv", index=False)

=== Memory-Safe Directed Link Prediction ===
Computing PageRank...
Extracting features...


100%|██████████| 200000/200000 [38:29<00:00, 86.60it/s] 


Dataset ready → 200,000 rows × 17 columns


In [5]:
#3


import random
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm


# ───────────────────────────────────────────────
# Balanced Directed Sampling
# ───────────────────────────────────────────────
def generate_directed_samples(G, num_samples=50_000, seed=42):
    random.seed(seed)

    nodes = list(G.nodes())
    edges = list(G.edges())

    pos_count = min(num_samples, len(edges))
    positive = random.sample(edges, pos_count)

    neg = set()
    while len(neg) < pos_count:
        u, v = random.sample(nodes, 2)
        if not G.has_edge(u, v) and not G.has_edge(v, u):
            neg.add((u, v))

    return positive, list(neg)


# ───────────────────────────────────────────────
# Lightweight Feature Extraction
# ───────────────────────────────────────────────
def extract_features(sample, G):
    u, v, label = sample

    features = {
        "source": u,
        "target": v,
        "label": label,
    }

    # Directed neighborhoods
    u_in = set(G.predecessors(u))
    u_out = set(G.successors(u))
    v_in = set(G.predecessors(v))
    v_out = set(G.successors(v))

    u_all = u_in | u_out
    v_all = v_in | v_out

    # ─── Degree Features ───
    features["u_d"] = len(u_all)
    features["u_d_in"] = len(u_in)
    features["u_d_out"] = len(u_out)
    features["u_d_bi"] = len(u_in & u_out)

    features["v_d"] = len(v_all)
    features["v_d_in"] = len(v_in)
    features["v_d_out"] = len(v_out)
    features["v_d_bi"] = len(v_in & v_out)

    # ─── Density Features ───
    features["u_in_density"] = len(u_in) / len(u_all) if len(u_all) > 0 else 0
    features["u_out_density"] = len(u_out) / len(u_all) if len(u_all) > 0 else 0
    features["v_in_density"] = len(v_in) / len(v_all) if len(v_all) > 0 else 0
    features["v_out_density"] = len(v_out) / len(v_all) if len(v_all) > 0 else 0

    # ─── Overlap Features ───
    common = u_all & v_all
    union = u_all | v_all

    features["common_friends"] = len(common)
    features["common_friends_in"] = len(u_in & v_in)
    features["common_friends_out"] = len(u_out & v_out)

    features["jaccard_coeff"] = len(common) / len(union) if len(union) > 0 else 0

    # ─── Transitive Features ───
    features["transitive_uv"] = len(u_out & v_in)
    features["transitive_vu"] = len(v_out & u_in)

    # ─── Structural Features ───
    features["pref_attachment"] = features["u_d"] * features["v_d"]
    features["opposite_direction"] = 1 if G.has_edge(v, u) else 0

    return features


# ───────────────────────────────────────────────
# Main Builder
# ───────────────────────────────────────────────
def build_directed_dataset_safe(
    G,
    num_samples=20_000,
    compute_pagerank=True,
    seed=42
):

    random.seed(seed)
    np.random.seed(seed)

    print("=== Memory-Safe Directed Link Prediction ===")

    # Sampling
    pos, neg = generate_directed_samples(G, num_samples, seed)
    samples = [(u, v, 1) for u, v in pos] + [(u, v, 0) for u, v in neg]
    random.shuffle(samples)

    # Optional PageRank
    pagerank = None
    if compute_pagerank:
        print("Computing PageRank...")
        pagerank = nx.pagerank(G, alpha=0.85)

    # Feature extraction
    rows = []
    print("Extracting features...")
    for sample in tqdm(samples):
        row = extract_features(sample, G)

        if pagerank:
            row["pr_source"] = pagerank.get(row["source"], 0)
            row["pr_target"] = pagerank.get(row["target"], 0)

        rows.append(row)

    df = pd.DataFrame(rows)

    print(f"Dataset ready → {df.shape[0]:,} rows × {df.shape[1]:,} columns")

    return df


# ───────────────────────────────────────────────
# Example Usage
# ───────────────────────────────────────────────
df = build_directed_dataset_safe(
    G_directed,
    num_samples=100_000,
    compute_pagerank=True,
    seed=42
)

df.to_csv("link_prediction_safe_additional.csv", index=False)

=== Memory-Safe Directed Link Prediction ===
Computing PageRank...
Extracting features...


100%|██████████| 200000/200000 [35:02<00:00, 95.12it/s] 


Dataset ready → 200,000 rows × 25 columns


In [ ]:
# import pandas as pd
# import networkx as nx
# import numpy as np
# import scipy.sparse as sp
# import random
# from tqdm import tqdm


# def build_rich_link_prediction_dataset(
#     G,                  # directed or undirected NetworkX graph
#     num_samples=100000, # number of positive AND negative samples each
#     use_undirected_for_topology=True,
#     partition=None,     # optional community/partition dict: node → community_id
#     seed=42
# ):
#     """
#     Creates a feature-rich dataset for link prediction.
    
#     Combines:
#     - Fast sparse-matrix based classical features (CN, JA, AA, PA, PageRank, degrees)
#     - Comprehensive social/link/subgraph features from the second implementation
#     - Works with directed graphs (falls back to undirected view for some metrics)
    
#     Returns:
#         pd.DataFrame with many topological/link/subgraph features + labels
#     """
#     random.seed(seed)
#     print(f"Building rich dataset — {num_samples:,} pos + {num_samples:,} neg samples")

#     # ───────────────────────────────────────────────
#     #  1. Prepare nodes, edges, mapping
#     # ───────────────────────────────────────────────
#     nodes = list(G.nodes())
#     directed = nx.is_directed(G)
    
#     node_to_idx = {node: i for i, node in enumerate(nodes)}
#     n = len(nodes)
    
#     # ───────────────────────────────────────────────
#     #  2. Sample positive and negative edges
#     # ───────────────────────────────────────────────
#     print("Sampling positive edges...")
#     existing_edges = list(G.edges())
#     pos_count = min(num_samples, len(existing_edges))
#     positive_samples = random.sample(existing_edges, pos_count)
    
#     print("Sampling negative (non-existent) edges...")
#     negative_samples = set()
#     while len(negative_samples) < pos_count:
#         u = random.choice(nodes)
#         v = random.choice(nodes)
#         if u != v and not G.has_edge(u, v):
#             negative_samples.add((u, v))
#     negative_samples = list(negative_samples)
    
#     # Combine
#     all_samples = [(u, v, 1) for u, v in positive_samples] + \
#                   [(u, v, 0) for u, v in negative_samples]
#     random.shuffle(all_samples)  # good practice
    
#     # ───────────────────────────────────────────────
#     #  3. Precompute global & fast-access structures
#     # ───────────────────────────────────────────────
#     print("Precomputing global metrics (PageRank, degrees)...")
#     pagerank_dict = nx.pagerank(G, alpha=0.85)
#     in_degree_dict  = dict(G.in_degree())
#     out_degree_dict = dict(G.out_degree())
    
#     # For topology features (common neighbors, AA, Jaccard) → usually use undirected view
#     if use_undirected_for_topology:
#         G_topo = G.to_undirected() if directed else G
#     else:
#         G_topo = G
    
#     A = nx.to_scipy_sparse_array(G_topo, nodelist=nodes, format='csr')
#     A_bin = A + A.T if not nx.is_directed(G_topo) else A
#     A_bin.data = np.ones_like(A_bin.data)   # binarize
    
#     degrees = np.array(A_bin.sum(axis=1)).flatten()           # undirected degrees
#     aa_weights = 1.0 / np.log(degrees + 2)                    # Adamic-Adar weights
    
#     # ───────────────────────────────────────────────
#     #  4. Feature extraction loop
#     # ───────────────────────────────────────────────
#     dataset = []
    
#     for u, v, label in tqdm(all_samples, desc="Extracting features"):
        
#         row = {
#             'source': u,
#             'target': v,
#             'label': label,
            
#             # Basic degrees (directed when available)
#             'out_deg_source': out_degree_dict.get(u, 0),
#             'in_deg_target':  in_degree_dict.get(v,  0),
#             'deg_u_undir':    degrees[node_to_idx[u]],
#             'deg_v_undir':    degrees[node_to_idx[v]],
            
#             # PageRank
#             'pr_source': pagerank_dict.get(u, 0),
#             'pr_target': pagerank_dict.get(v, 0),
#         }
        
#         idx_u = node_to_idx[u]
#         idx_v = node_to_idx[v]
        
#         # ─── Sparse fast classical features ──────────────────────────────
#         row_u = A_bin[idx_u]
#         row_v = A_bin[idx_v]
        
#         inter = row_u.multiply(row_v)
#         cn = inter.sum()
        
#         union_size = degrees[idx_u] + degrees[idx_v] - cn
#         jaccard = cn / union_size if union_size > 0 else 0.0
        
#         aa = np.sum(aa_weights[inter.indices]) if cn > 0 else 0.0
        
#         pref = degrees[idx_u] * degrees[idx_v]
        
#         row.update({
#             'common_neighbors': cn,
#             'jaccard':          jaccard,
#             'adamic_adar':      aa,
#             'pref_attachment':  pref,
#         })
        
#         # ─── Neighborhood sets (for more expressive features) ────────────
#         nbr_u = set(G_topo.neighbors(u))
#         nbr_v = set(G_topo.neighbors(v))
        
#         common_nbrs = nbr_u & nbr_v
#         union_nbrs  = nbr_u | nbr_v
        
#         # ─── Vertex / ego features ───────────────────────────────────────
#         row['d_u'] = len(nbr_u)
#         row['d_v'] = len(nbr_v)
        
#         u_sub = G_topo.subgraph(nbr_u)
#         v_sub = G_topo.subgraph(nbr_v)
        
#         row['u_ego_links']     = u_sub.number_of_edges()
#         row['v_ego_links']     = v_sub.number_of_edges()
#         row['u_ego_density']   = row['u_ego_links'] / (row['d_u'] * (row['d_u']-1)/2) if row['d_u'] > 1 else 0
#         row['v_ego_density']   = row['v_ego_links'] / (row['d_v'] * (row['d_v']-1)/2) if row['d_v'] > 1 else 0
        
#         # ─── Link / neighborhood overlap features ────────────────────────
#         row['common_friends']   = len(common_nbrs)
#         row['union_friends']    = len(union_nbrs)
#         row['jaccard_coeff']    = row['common_friends'] / row['union_friends'] if row['union_friends'] > 0 else 0
        
#         # Friends Measure (common + friends-of-friends connections)
#         cross_links = sum(1 for x in nbr_u for y in nbr_v if G_topo.has_edge(x, y))
#         row['friends_measure'] = cross_links + len(common_nbrs)  # includes common neighbors
        
#         if partition is not None:
#             row['same_community'] = 1 if partition.get(u) == partition.get(v) else 0
#         else:
#             row['same_community'] = 0
        
#         # ─── Link subgraph features ──────────────────────────────────────
#         link_subgraph = G_topo.subgraph(union_nbrs)
#         row['link_subgraph_edges'] = link_subgraph.number_of_edges()
        
#         try:
#             row['link_subgraph_wcc'] = nx.number_connected_components(link_subgraph)
#         except:
#             row['link_subgraph_wcc'] = -1
        
#         row['inner_cross_links'] = cross_links  # friends of u connected to friends of v (excluding commons)
        
#         # ─── Path features ───────────────────────────────────────────────
#         try:
#             row['shortest_path_length'] = nx.shortest_path_length(G_topo, u, v)
#         except nx.NetworkXNoPath:
#             row['shortest_path_length'] = -1
        
#         dataset.append(row)
    
#     print("Feature extraction finished.")
#     df = pd.DataFrame(dataset)
    
#     # Optional: nicer column order
#     cols_first = ['source', 'target', 'label',
#                   'common_neighbors', 'jaccard', 'adamic_adar', 'pref_attachment',
#                   'pr_source', 'pr_target', 'out_deg_source', 'in_deg_target']
    
#     other_cols = [c for c in df.columns if c not in cols_first]
#     df = df[cols_first + other_cols]
    
#     return df


# # ───────────────────────────────────────────────
# #  Example usage
# # ───────────────────────────────────────────────
# df = build_rich_link_prediction_dataset(
#     G_directed,
#     num_samples=50_000,
#     use_undirected_for_topology=True,
#     partition=None,   # or your community dict
# )

# df.to_csv("link_prediction_rich_features.csv", index=False)
# print(df.shape)
# print(df.head(2))

































# import random
# import pandas as pd
# import numpy as np
# import networkx as nx
# import scipy.sparse as sp
# from multiprocessing import Pool, cpu_count
# from functools import partial
# from tqdm import tqdm
# import community as community_louvain  # python-louvain / community


# def generate_directed_samples(G, num_samples=100_000, seed=42):
#     """
#     Generate balanced positive and negative directed edge samples.
#     Negative samples do NOT exist in either direction.
#     """
#     random.seed(seed)
#     print(f"Generating {num_samples:,} positive + {num_samples:,} negative directed samples...")

#     nodes = list(G.nodes())
#     all_edges = list(G.edges())

#     # Positive samples (existing directed edges)
#     pos_count = min(num_samples, len(all_edges))
#     positive_samples = random.sample(all_edges, pos_count)

#     # Negative samples (neither (u,v) nor (v,u) exists)
#     print("Sampling negative edges (both directions must be absent)...")
#     neg_samples = set()
#     while len(neg_samples) < pos_count:
#         u, v = random.sample(nodes, 2)
#         if not G.has_edge(u, v) and not G.has_edge(v, u):
#             neg_samples.add((u, v))

#     print("Sampling complete.")
#     return positive_samples, list(neg_samples)


# def extract_all_features_single(sample, G, partition=None, G_undir=None, A_bin=None, degrees_undir=None, aa_weights=None):
#     """
#     Extract rich directed + classical features for one (u, v, label) triple.
#     Designed to be called in parallel.
#     """
#     u, v, label = sample

#     features = {
#         'source': u,
#         'target': v,
#         'label': label,
#     }

#     try:
#         # ─── Neighborhoods ───────────────────────────────────────────────
#         u_in   = set(G.predecessors(u))
#         u_out  = set(G.successors(u))
#         v_in   = set(G.predecessors(v))
#         v_out  = set(G.successors(v))

#         u_all  = u_in | u_out
#         v_all  = v_in | v_out

#         # ─── Vertex features (both u and v) ──────────────────────────────
#         for node, n_in, n_out, n_all, prefix in [
#             (u, u_in, u_out, u_all, 'u'),
#             (v, v_in, v_out, v_all, 'v')
#         ]:
#             d     = len(n_all)
#             d_in  = len(n_in)
#             d_out = len(n_out)
#             d_bi  = len(n_in & n_out)

#             features[f'{prefix}_d']       = d
#             features[f'{prefix}_d_in']    = d_in
#             features[f'{prefix}_d_out']   = d_out
#             features[f'{prefix}_d_bi']    = d_bi

#             features[f'{prefix}_in_density']  = d_in  / d if d > 0 else 0
#             features[f'{prefix}_out_density'] = d_out / d if d > 0 else 0
#             features[f'{prefix}_bi_density']  = d_bi  / d if d > 0 else 0

#             # Neighborhood subgraphs
#             nh_sub      = G.subgraph(n_all)
#             nh_sub_plus = G.subgraph(n_all | {node})

#             features[f'{prefix}_nh_links']       = nh_sub.number_of_edges()
#             features[f'{prefix}_nh_plus_links']  = nh_sub_plus.number_of_edges()

#             features[f'{prefix}_nh_density']      = d / features[f'{prefix}_nh_links']      if features[f'{prefix}_nh_links']      > 0 else 0
#             features[f'{prefix}_nh_plus_density'] = d / features[f'{prefix}_nh_plus_links'] if features[f'{prefix}_nh_plus_links'] > 0 else 0

#             # Connectivity
#             scc      = nx.number_strongly_connected_components(nh_sub)      if len(nh_sub)      > 0 else 1
#             wcc      = nx.number_weakly_connected_components(nh_sub)      if len(nh_sub)      > 0 else 1
#             scc_plus = nx.number_strongly_connected_components(nh_sub_plus) if len(nh_sub_plus) > 0 else 1

#             features[f'{prefix}_scc']        = scc
#             features[f'{prefix}_wcc']        = wcc
#             features[f'{prefix}_scc_plus']   = scc_plus
#             features[f'{prefix}_avg_scc']     = d / scc
#             features[f'{prefix}_avg_wcc']     = d / wcc
#             features[f'{prefix}_avg_scc_plus']= d / scc_plus

#         # ─── Link / overlap features ─────────────────────────────────────
#         common_all = u_all & v_all
#         total_all  = u_all | v_all

#         features['common_friends']      = len(common_all)
#         features['common_friends_in']   = len(u_in & v_in)
#         features['common_friends_out']  = len(u_out & v_out)
#         features['common_friends_bi']   = len((u_in & u_out) & (v_in & v_out))
#         features['total_friends']       = len(total_all)
#         features['jaccard_coeff']       = features['common_friends'] / features['total_friends'] if features['total_friends'] > 0 else 0

#         features['transitive_friends_uv'] = len(u_out & v_in)   # u → x → v
#         features['transitive_friends_vu'] = len(v_out & u_in)   # v → x → u

#         features['pref_attachment'] = features['u_d'] * features['v_d']

#         # Friends measure (common + direct cross edges)
#         cross = sum(1 for x in u_all for y in v_all if G.has_edge(x, y) or G.has_edge(y, x))
#         features['friends_measure'] = cross + features['common_friends']

#         features['opposite_direction_friends'] = 1 if G.has_edge(v, u) else 0

#         if partition is not None:
#             features['same_community'] = 1 if partition.get(u) == partition.get(v) else 0
#         else:
#             features['same_community'] = 0

#         # ─── Link subgraph features ──────────────────────────────────────
#         link_sub      = G.subgraph(total_all)
#         link_sub_plus = G.subgraph(total_all | {u, v})

#         features['link_subgraph_edges']      = link_sub.number_of_edges()
#         features['link_subgraph_plus_edges'] = link_sub_plus.number_of_edges()
#         features['inner_subgraph_edges']     = cross   # cross edges excluding commons

#         features['link_subgraph_scc']      = nx.number_strongly_connected_components(link_sub)      if len(link_sub)      > 0 else 0
#         features['link_subgraph_wcc']      = nx.number_weakly_connected_components(link_sub)      if len(link_sub)      > 0 else 0
#         features['link_subgraph_plus_scc'] = nx.number_strongly_connected_components(link_sub_plus) if len(link_sub_plus) > 0 else 0

#         # ─── Path features ───────────────────────────────────────────────
#         try:    features['shortest_path_uv'] = nx.shortest_path_length(G, u, v)
#         except: features['shortest_path_uv'] = -1

#         try:    features['shortest_path_vu'] = nx.shortest_path_length(G, v, u)
#         except: features['shortest_path_vu'] = -1

#         # ─── Fast classical undirected-view features ─────────────────────
#         if A_bin is not None and degrees_undir is not None:
#             idx_u = node_to_idx[u]
#             idx_v = node_to_idx[v]

#             row_u = A_bin[idx_u]
#             row_v = A_bin[idx_v]

#             inter = row_u.multiply(row_v)
#             cn    = inter.sum()

#             union_size = degrees_undir[idx_u] + degrees_undir[idx_v] - cn
#             ja    = cn / union_size if union_size > 0 else 0.0
#             aa    = np.sum(aa_weights[inter.indices]) if cn > 0 else 0.0
#             pa    = degrees_undir[idx_u] * degrees_undir[idx_v]

#             features.update({
#                 'common_neighbors_undir': cn,
#                 'jaccard_undir':          ja,
#                 'adamic_adar_undir':      aa,
#                 'pref_attachment_undir':  pa,
#             })

#     except Exception as e:
#         # Isolated node / other edge-case → keep going with defaults
#         pass

#     return features


# def build_rich_directed_link_prediction_dataset(
#     G,
#     num_samples=100_000,
#     compute_louvain=True,
#     use_multiprocessing=True,
#     n_jobs=None,
#     seed=42
# ):
#     """
#     End-to-end function:
#       1. Optional Louvain partitioning
#       2. Balanced directed sampling
#       3. Precompute sparse undirected topology structures
#       4. Extract very rich feature set (directed + classical)
#     """
#     random.seed(seed)
#     np.random.seed(seed)

#     print("=== Rich Directed Link Prediction Feature Extraction ===")

#     # 1. Optional community detection
#     partition = None
#     if compute_louvain:
#         print("Running Louvain community detection (undirected view)...")
#         G_undir = G.to_undirected()
#         partition = community_louvain.best_partition(G_undir)

#     # 2. Generate samples
#     pos_samples, neg_samples = generate_directed_samples(G, num_samples, seed)
#     all_samples = [(u,v,1) for u,v in pos_samples] + [(u,v,0) for u,v in neg_samples]
#     random.shuffle(all_samples)

#     # 3. Precompute fast undirected topology structures
#     print("Precomputing undirected topology structures (CN, AA, JA)...")
#     nodes = list(G.nodes())
#     global node_to_idx
#     node_to_idx = {n: i for i, n in enumerate(nodes)}

#     G_undir = G.to_undirected()
#     A = nx.to_scipy_sparse_array(G_undir, nodelist=nodes, format='csr')
#     A_bin = A + A.T
#     A_bin.data = np.ones_like(A_bin.data)

#     degrees_undir = np.array(A_bin.sum(axis=1)).flatten()
#     aa_weights = 1.0 / np.log(degrees_undir + 2)

#     # PageRank (directed)
#     print("Computing PageRank...")
#     pagerank = nx.pagerank(G, alpha=0.85)

#     # 4. Feature extraction (parallel or sequential)
#     print("Extracting features...")
#     func = partial(extract_all_features_single,
#                    G=G,
#                    partition=partition,
#                    G_undir=G_undir,
#                    A_bin=A_bin,
#                    degrees_undir=degrees_undir,
#                    aa_weights=aa_weights)

#     if use_multiprocessing:
#         n_jobs = n_jobs or max(1, cpu_count() - 1)
#         print(f"Using multiprocessing with {n_jobs} workers...")
#         with Pool(n_jobs) as pool:
#             results = list(tqdm(
#                 pool.imap(func, all_samples),
#                 total=len(all_samples),
#                 desc="Feature extraction"
#             ))
#     else:
#         results = [func(s) for s in tqdm(all_samples, desc="Feature extraction")]

#     # 5. Build DataFrame + add PageRank
#     print("Building final DataFrame...")
#     df = pd.DataFrame(results)

#     # Add PageRank
#     df['pr_source'] = df['source'].map(pagerank).fillna(0)
#     df['pr_target'] = df['target'].map(pagerank).fillna(0)

#     # Nice column order
#     cols_first = [
#         'source', 'target', 'label',
#         'common_neighbors_undir', 'jaccard_undir', 'adamic_adar_undir', 'pref_attachment_undir',
#         'pr_source', 'pr_target',
#         'u_d', 'u_d_in', 'u_d_out', 'v_d', 'v_d_in', 'v_d_out',
#         'common_friends', 'common_friends_in', 'common_friends_out',
#         'jaccard_coeff', 'transitive_friends_uv', 'transitive_friends_vu',
#         'shortest_path_uv', 'shortest_path_vu'
#     ]

#     cols_first = [c for c in cols_first if c in df.columns]
#     other_cols = [c for c in df.columns if c not in cols_first]
#     df = df[cols_first + other_cols]

#     print(f"Dataset ready → {df.shape[0]:,} rows × {df.shape[1]:,} columns")
#     return df


# # ───────────────────────────────────────────────
# #  Example usage
# # ───────────────────────────────────────────────
# df = build_rich_directed_link_prediction_dataset(
#     G_directed,
#     num_samples=50_000,
#     compute_louvain=True,
#     use_multiprocessing=True,
#     n_jobs=None,           # auto-detect
#     seed=42
# )


# df.to_csv("link_prediction_rich_directed_combined.csv", index=False)

In [ ]:
# import random
# import pandas as pd
# import numpy as np
# import networkx as nx
# from multiprocessing import Pool, cpu_count
# from functools import partial
# from tqdm import tqdm

# # Safe Louvain import
# try:
#     import community.community_louvain as community_louvain
#     LOUVAIN_AVAILABLE = True
# except ImportError:
#     print("Louvain not available. Community feature will be disabled.")
#     LOUVAIN_AVAILABLE = False


# # ───────────────────────────────────────────────
# # Balanced Directed Sampling
# # ───────────────────────────────────────────────
# def generate_directed_samples(G, num_samples=100_000, seed=42):
#     random.seed(seed)

#     nodes = list(G.nodes())
#     all_edges = list(G.edges())

#     pos_count = min(num_samples, len(all_edges))
#     positive_samples = random.sample(all_edges, pos_count)

#     neg_samples = set()
#     while len(neg_samples) < pos_count:
#         u, v = random.sample(nodes, 2)
#         if not G.has_edge(u, v) and not G.has_edge(v, u):
#             neg_samples.add((u, v))

#     return positive_samples, list(neg_samples)


# # ───────────────────────────────────────────────
# # Fast Feature Extraction (Scalable Version)
# # ───────────────────────────────────────────────
# def extract_fast_features(sample, G, partition,
#                           A_bin, degrees_undir, aa_weights,
#                           node_to_idx):
#     u, v, label = sample

#     features = {
#         'source': u,
#         'target': v,
#         'label': label,
#     }

#     # Directed neighborhoods
#     u_in  = set(G.predecessors(u))
#     u_out = set(G.successors(u))
#     v_in  = set(G.predecessors(v))
#     v_out = set(G.successors(v))

#     u_all = u_in | u_out
#     v_all = v_in | v_out

#     # Degree features
#     for node, n_in, n_out, n_all, prefix in [
#         (u, u_in, u_out, u_all, 'u'),
#         (v, v_in, v_out, v_all, 'v')
#     ]:
#         d     = len(n_all)
#         d_in  = len(n_in)
#         d_out = len(n_out)
#         d_bi  = len(n_in & n_out)

#         features[f'{prefix}_d']     = d
#         features[f'{prefix}_d_in']  = d_in
#         features[f'{prefix}_d_out'] = d_out
#         features[f'{prefix}_d_bi']  = d_bi

#         features[f'{prefix}_in_density']  = d_in / d if d > 0 else 0
#         features[f'{prefix}_out_density'] = d_out / d if d > 0 else 0
#         features[f'{prefix}_bi_density']  = d_bi / d if d > 0 else 0

#     # Overlap features
#     common_all = u_all & v_all
#     total_all  = u_all | v_all

#     features['common_friends']     = len(common_all)
#     features['common_friends_in']  = len(u_in & v_in)
#     features['common_friends_out'] = len(u_out & v_out)
#     features['common_friends_bi']  = len((u_in & u_out) & (v_in & v_out))
#     features['total_friends']      = len(total_all)

#     features['jaccard_coeff'] = (
#         len(common_all) / len(total_all)
#         if len(total_all) > 0 else 0
#     )

#     features['transitive_friends_uv'] = len(u_out & v_in)
#     features['transitive_friends_vu'] = len(v_out & u_in)

#     features['pref_attachment'] = features['u_d'] * features['v_d']
#     features['opposite_direction_friends'] = 1 if G.has_edge(v, u) else 0

#     # Community feature
#     if partition is not None:
#         features['same_community'] = (
#             1 if partition.get(u) == partition.get(v) else 0
#         )
#     else:
#         features['same_community'] = 0

#     # Undirected sparse features
#     idx_u = node_to_idx[u]
#     idx_v = node_to_idx[v]

#     row_u = A_bin[idx_u]
#     row_v = A_bin[idx_v]

#     inter = row_u.multiply(row_v)
#     cn    = inter.sum()

#     union_size = degrees_undir[idx_u] + degrees_undir[idx_v] - cn
#     ja    = cn / union_size if union_size > 0 else 0.0
#     aa    = np.sum(aa_weights[inter.indices]) if cn > 0 else 0.0
#     pa    = degrees_undir[idx_u] * degrees_undir[idx_v]

#     features.update({
#         'common_neighbors_undir': cn,
#         'jaccard_undir':          ja,
#         'adamic_adar_undir':      aa,
#         'pref_attachment_undir':  pa,
#     })

#     return features


# # ───────────────────────────────────────────────
# # Main Builder
# # ───────────────────────────────────────────────
# def build_fast_directed_link_prediction_dataset(
#     G,
#     num_samples=100_000,
#     compute_louvain=True,
#     use_multiprocessing=True,
#     n_jobs=None,
#     seed=42
# ):

#     random.seed(seed)
#     np.random.seed(seed)

#     print("=== Fast Directed Link Prediction ===")

#     # Community detection
#     partition = None
#     if compute_louvain and LOUVAIN_AVAILABLE:
#         print("Running Louvain...")
#         partition = community_louvain.best_partition(G.to_undirected())

#     # Sampling
#     pos_samples, neg_samples = generate_directed_samples(G, num_samples, seed)
#     all_samples = [(u, v, 1) for u, v in pos_samples] + \
#                   [(u, v, 0) for u, v in neg_samples]
#     random.shuffle(all_samples)

#     # Sparse structures
#     print("Precomputing sparse structures...")
#     nodes = list(G.nodes())
#     node_to_idx = {n: i for i, n in enumerate(nodes)}

#     G_undir = G.to_undirected()
#     A = nx.to_scipy_sparse_array(G_undir, nodelist=nodes, format='csr')
#     A_bin = A + A.T
#     A_bin.data = np.ones_like(A_bin.data)

#     degrees_undir = np.array(A_bin.sum(axis=1)).flatten()
#     aa_weights = 1.0 / np.log(degrees_undir + 2)

#     # PageRank
#     print("Computing PageRank...")
#     pagerank = nx.pagerank(G, alpha=0.85)

#     # Feature extraction
#     print("Extracting features...")

#     func = partial(
#         extract_fast_features,
#         G=G,
#         partition=partition,
#         A_bin=A_bin,
#         degrees_undir=degrees_undir,
#         aa_weights=aa_weights,
#         node_to_idx=node_to_idx
#     )

#     if use_multiprocessing:
#         n_jobs = n_jobs or max(1, cpu_count() - 1)
#         with Pool(n_jobs) as pool:
#             results = list(tqdm(
#                 pool.imap(func, all_samples),
#                 total=len(all_samples),
#                 desc="Feature extraction"
#             ))
#     else:
#         results = [func(s) for s in tqdm(all_samples)]

#     df = pd.DataFrame(results)

#     df['pr_source'] = df['source'].map(pagerank).fillna(0)
#     df['pr_target'] = df['target'].map(pagerank).fillna(0)

#     print(f"Dataset ready → {df.shape[0]:,} rows × {df.shape[1]:,} columns")
#     return df


# # ───────────────────────────────────────────────
# # Example Usage
# # ───────────────────────────────────────────────
# if __name__ == "__main__":

#     df = build_fast_directed_link_prediction_dataset(
#         G_directed,
#         num_samples=50_000,
#         compute_louvain=True,
#         use_multiprocessing=True,
#         seed=42
#     )

#     df.to_csv("link_prediction_fast_directed.csv", index=False)